# Formativa 2 — Fase 3: modelamiento predictivo de `RainTomorrow`

**Grupo 7 · Proyecto Lluvias Australia**

## Objetivo y decisiones metodológicas

Este notebook integra hallazgos de S1/S2 y prepara un clasificador para estimar `RainTomorrow`. Compara eliminación de filas, imputación simple e imputación por regresión de `Sunshine`; realiza una partición estratificada 70/30; calcula VIF; ajusta tres regresiones logísticas (variables justificadas, stepwise AIC y modelo reducido); evalúa matriz de confusión, accuracy, precisión, recall, F1, ROC-AUC y PR-AUC; y aplica bootstrap por localidad a los coeficientes del modelo seleccionado.

La clase positiva representa cerca del 22,42 % de los registros etiquetados, por lo que la exactitud no basta. Se pondera `RainTomorrow = Yes` con peso 2,2 para priorizar su detección. Un falso negativo significa no anticipar lluvia que sí ocurre; un falso positivo anticipa lluvia que no ocurre. La importancia relativa depende del uso previsto.

**Limitaciones:** el CSV ya fue explorado en S1/S2; por ello, el test no es totalmente independiente de decisiones exploratorias anteriores. La partición aleatoria estratificada mezcla fechas y localidades, así que no prueba por sí sola la generalización a años futuros. Los coeficientes expresan asociaciones, no causalidad.

## 1. Entorno y carga de datos
Se conserva intacto el CSV original. Ejecuta el notebook desde la raíz del repositorio.

In [9]:
from pathlib import Path
import json, warnings, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, precision_recall_curve
)
from IPython.display import display, Markdown
warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 2026
np.random.seed(SEED)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)

candidatos = [Path("data/raw/weatherAUS.csv"), Path("../data/raw/weatherAUS.csv")]
candidatos += [p / "data/raw/weatherAUS.csv" for p in [Path.cwd(), *Path.cwd().parents]]
ruta = next((p.resolve() for p in candidatos if p.is_file()), None)
if ruta is None:
    raise FileNotFoundError("No se encontró data/raw/weatherAUS.csv; ejecuta desde la raíz del repositorio.")
df_original = pd.read_csv(ruta, parse_dates=["Date"])
df = df_original[df_original["RainTomorrow"].isin(["Yes", "No"])].copy()
df["y"] = df["RainTomorrow"].map({"No": 0, "Yes": 1}).astype(int)
print("Archivo:", ruta, "| original:", df_original.shape, "| con etiqueta:", df.shape)
display(df[["RainTomorrow", "y"]].value_counts().rename("n").to_frame())
print(f"Prevalencia de Yes: {df.y.mean():.2%}")

ModuleNotFoundError: No module named 'numpy'

## 2. Distribución de clases y faltantes

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df["RainTomorrow"].value_counts().reindex(["No", "Yes"])
ax.bar(counts.index, counts.values)
ax.set(title="Distribución de RainTomorrow", xlabel="Lluvia al día siguiente", ylabel="Observaciones")
plt.tight_layout(); plt.show()

features_valid = ["Humidity3pm", "Rainfall", "Sunshine", "MaxTemp", "RainToday"]
candidatos_aic = ["Humidity3pm", "Rainfall", "Sunshine", "Pressure3pm", "MaxTemp", "RainToday"]
reducido = ["Humidity3pm", "Rainfall", "RainToday"]
display((100 * df[features_valid].isna().mean()).sort_values(ascending=False).rename("% faltante").to_frame().round(2))

## 3. Selección de variables

S1 encontró asociaciones entre `RainTomorrow` y `Humidity3pm`, `Sunshine`, `Rainfall`, `Pressure3pm`, además de una diferencia importante según `RainToday`. La correlación entre `MaxTemp` y `Temp3pm` fue aproximadamente 0,984, por lo que no se incluyen juntas en los modelos. El modelo principal utiliza cinco predictores para respetar la pauta de 3–5 variables; `Pressure3pm` se considera en el conjunto candidato del stepwise. Estas asociaciones no implican causalidad.

## 4. Partición estratificada 70/30

In [ ]:
train_idx, test_idx = train_test_split(
    df.index, test_size=0.30, random_state=SEED, stratify=df["y"]
)
train = df.loc[train_idx].copy()
test = df.loc[test_idx].copy()
display(pd.DataFrame({
    "conjunto": ["Entrenamiento", "Prueba"],
    "n": [len(train), len(test)],
    "% Yes": [100*train.y.mean(), 100*test.y.mean()]
}).round(3))
assert set(train.index).isdisjoint(set(test.index))

## 5. Comparación de estrategias de datos faltantes

Se comparan tres estrategias con los mismos cinco predictores y la misma partición. Para que la evaluación sea comparable, las tres se miden sobre los mismos casos completos del conjunto de prueba; esto limita la conclusión a esa cohorte y no refleja toda la cobertura de los métodos.

1. **Eliminación:** entrena solo con filas completas.
2. **Imputación simple:** mediana para variables numéricas y categoría más frecuente para `RainToday`.
3. **Imputación por regresión:** `Sunshine` se estima con un Random Forest entrenado únicamente en entrenamiento y en filas donde `Sunshine` está observado. Los demás faltantes se imputan con mediana/moda.

El 48 % aproximado de faltantes en `Sunshine`, incluyendo localidades sin mediciones observadas, limita la certeza de las imputaciones. No se ha demostrado que el mecanismo de ausencia sea MCAR o MAR.

In [ ]:
num_features = ["Humidity3pm", "Rainfall", "Sunshine", "MaxTemp"]
cat_features = ["RainToday"]
test_complete = test.dropna(subset=features_valid).copy()
train_complete = train.dropna(subset=features_valid).copy()
print(f"Filas completas de entrenamiento: {len(train_complete):,}/{len(train):,}")
print(f"Cohorte común de test: {len(test_complete):,}/{len(test):,}")

def preprocessor(features):
    nums = [c for c in features if c != "RainToday"]
    cats = [c for c in features if c == "RainToday"]
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), nums),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", drop="if_binary"))]), cats)
    ])

def sklearn_model(features):
    return Pipeline([
        ("prep", preprocessor(features)),
        ("logit", LogisticRegression(class_weight={0: 1.0, 1: 2.2}, max_iter=1500, random_state=SEED))
    ])

# Imputación simple: todos los imputadores se ajustan solo en train.
m_simple = sklearn_model(features_valid).fit(train[features_valid], train.y)

# Imputación por regresión de Sunshine. No se usa RainTomorrow para estimarla.
sun_num = ["MinTemp", "MaxTemp", "Rainfall", "Evaporation", "WindGustSpeed",
           "Humidity9am", "Humidity3pm", "Pressure9am", "Pressure3pm",
           "Cloud9am", "Cloud3pm", "Temp9am", "WindSpeed9am", "WindSpeed3pm"]
sun_cat = ["Location", "RainToday"]
sun_features = sun_num + sun_cat
sun_train = train[train["Sunshine"].notna()]
sun_reg = Pipeline([
    ("prep", ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), sun_num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), sun_cat)
    ])),
    ("rf", RandomForestRegressor(n_estimators=60, max_depth=14, min_samples_leaf=5,
                                 n_jobs=-1, random_state=SEED))
]).fit(sun_train[sun_features], sun_train["Sunshine"])

def imputar_sunshine(data):
    out = data.copy()
    mask = out["Sunshine"].isna()
    if mask.any():
        out.loc[mask, "Sunshine"] = sun_reg.predict(out.loc[mask, sun_features])
    return out

train_reg = imputar_sunshine(train)
test_reg = imputar_sunshine(test)
m_reg = sklearn_model(features_valid).fit(train_reg[features_valid], train_reg.y)

m_del = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), cat_features)
    ])),
    ("logit", LogisticRegression(class_weight={0: 1.0, 1: 2.2}, max_iter=1500, random_state=SEED))
]).fit(train_complete[features_valid], train_complete.y)

def metricas(y, p, umbral=0.5):
    pred = (np.asarray(p) >= umbral).astype(int)
    return {"n_test_comun": len(y), "accuracy": accuracy_score(y, pred),
            "precision_yes": precision_score(y, pred, zero_division=0),
            "recall_yes": recall_score(y, pred, zero_division=0),
            "f1_yes": f1_score(y, pred, zero_division=0),
            "roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p)}

y_comun = test_complete.y.to_numpy()
probs_imp = {
    "Eliminación de filas": m_del.predict_proba(test_complete[features_valid])[:, 1],
    "Imputación simple": m_simple.predict_proba(test_complete[features_valid])[:, 1],
    "Regresión de Sunshine": m_reg.predict_proba(test_complete[features_valid])[:, 1]
}
comparacion_imputacion = pd.DataFrame([
    {"estrategia": name, **metricas(y_comun, prob)} for name, prob in probs_imp.items()
]).set_index("estrategia")
display(comparacion_imputacion.round(4))

## 6. VIF y multicolinealidad
El VIF se calcula en predictores numéricos completos de entrenamiento. `Temp3pm` se incluye solo para diagnóstico, no junto con `MaxTemp` en los modelos. Valores altos sugieren redundancia lineal; no constituyen por sí solos una regla automática para eliminar variables.

In [ ]:
vif_vars = ["MinTemp", "MaxTemp", "Temp3pm", "Humidity3pm", "Rainfall", "Pressure3pm", "Sunshine"]
vif_data = train[vif_vars].dropna()
if len(vif_data) > 30000:
    vif_data = vif_data.sample(30000, random_state=SEED)
Xv = sm.add_constant(vif_data.astype(float), has_constant="add")
vif = pd.DataFrame({"variable": Xv.columns,
                    "VIF": [variance_inflation_factor(Xv.values, i) for i in range(Xv.shape[1])]})
vif = vif[vif.variable != "const"].sort_values("VIF", ascending=False)
display(vif.round(3))

## 7. Regresiones logísticas: variables justificadas, stepwise AIC y modelo reducido

El stepwise usa exclusivamente entrenamiento y compara AIC de modelos sin ponderar. Los tres modelos predictivos finales se ajustan con peso 2,2 para la clase positiva. Los coeficientes, odds ratios y p-valores se interpretan con cautela: el ponderador cambia el ajuste y la dependencia temporal/espacial puede afectar la inferencia.

In [ ]:
MEDIANAS_ENTRENAMIENTO = {
    c: pd.to_numeric(train_reg[c], errors="coerce").median()
    for c in candidatos_aic
    if c != "RainToday"
}

MODA_RAIN_TODAY = train_reg["RainToday"].mode(
    dropna=True
).iloc[0]


def sm_data(data, features):
    X = data[features].copy()

    for c in X.columns:
        if c == "RainToday":
            X[c] = X[c].fillna(MODA_RAIN_TODAY)
            X[c] = X[c].map({"No": 0, "Yes": 1})
        else:
            X[c] = pd.to_numeric(X[c], errors="coerce")
            X[c] = X[c].fillna(MEDIANAS_ENTRENAMIENTO[c])

    X = sm.add_constant(X.astype(float), has_constant="add")
    y = data["y"].astype(int)

    return X, y

def fit_aic(data, features):
    X, y = sm_data(data, features)
    return sm.GLM(y, X, family=sm.families.Binomial()).fit()

def stepwise_aic(data, candidates, initial):
    selected = list(initial)
    model = fit_aic(data, selected)
    hist = [{"paso": 0, "acción": "inicio", "variable": ", ".join(selected), "AIC": model.aic}]
    while True:
        options = []
        for v in candidates:
            if v not in selected:
                trial = selected + [v]
                try: options.append((fit_aic(data, trial).aic, "añadir", v, trial))
                except Exception: pass
        if len(selected) > 2:
            for v in selected:
                trial = [x for x in selected if x != v]
                try: options.append((fit_aic(data, trial).aic, "retirar", v, trial))
                except Exception: pass
        if not options: break
        aic_new, action, variable, trial = min(options, key=lambda z: z[0])
        if aic_new >= model.aic - 1e-6: break
        selected = trial
        model = fit_aic(data, selected)
        hist.append({"paso": len(hist), "acción": action, "variable": variable, "AIC": model.aic})
    return selected, model, pd.DataFrame(hist)

step_vars, step_model_unweighted, aic_history = stepwise_aic(
    train_reg, candidatos_aic, ["Humidity3pm", "RainToday"]
)
display(aic_history.round(2))
print("Variables seleccionadas por AIC:", step_vars)

specs = {
    "A · Variables justificadas": features_valid,
    "B · Stepwise AIC": step_vars,
    "C · Reducido": reducido
}

def fit_weighted(data, features):
    X, y = sm_data(data, features)
    weights = np.where(y.to_numpy() == 1, 2.2, 1.0)
    return sm.GLM(y, X, family=sm.families.Binomial(), freq_weights=weights).fit()

models = {name: fit_weighted(train_reg, vars_) for name, vars_ in specs.items()}
aic_table = pd.DataFrame([
    {"modelo": name, "predictores": ", ".join(vars_), "n_predictores": len(vars_),
     "AIC_sin_ponderar": fit_aic(train_reg, vars_).aic}
    for name, vars_ in specs.items()
])
display(aic_table.round(2))

In [ ]:
def coef_table(model, name):
    tab = pd.DataFrame({
        "coeficiente": model.params, "error_std": model.bse, "z": model.tvalues,
        "p_valor": model.pvalues, "OR": np.exp(model.params),
        "OR_IC95_inf": np.exp(model.conf_int()[0]),
        "OR_IC95_sup": np.exp(model.conf_int()[1])
    })
    tab.insert(0, "modelo", name)
    return tab

coeficientes = pd.concat([coef_table(m, name) for name, m in models.items()])
display(coeficientes.round(4))

## 8. Evaluación en test: matriz de confusión, métricas y curvas

El umbral 0,50 se utiliza como referencia común; no se optimiza sobre test. En la fase siguiente, el umbral operativo debe elegirse mediante validación dentro del conjunto de entrenamiento, de acuerdo con el equilibrio deseado entre recall y precisión.

In [ ]:
def predict_sm(model, data, features):
    X, _ = sm_data(data, features)
    return np.asarray(model.predict(X))

probs_test, preds_test, rows = {}, {}, []
for name, features in specs.items():
    p = predict_sm(models[name], test_reg, features)
    pred = (p >= 0.5).astype(int)
    probs_test[name], preds_test[name] = p, pred
    rows.append({"modelo": name, "n_test": len(test_reg), **metricas(test_reg.y, p)})
metricas_test = pd.DataFrame(rows).set_index("modelo")
display(metricas_test.round(4))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, _) in zip(axes, specs.items()):
    cm = confusion_matrix(test_reg.y, preds_test[name], labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=["No", "Yes"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
for name, p in probs_test.items():
    fpr, tpr, _ = roc_curve(test_reg.y, p)
    ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(test_reg.y, p):.3f})")
ax.plot([0, 1], [0, 1], "--", label="Referencia aleatoria")
ax.set(xlabel="Tasa de falsos positivos", ylabel="Recall", title="Curvas ROC — test")
ax.legend(); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
for name, p in probs_test.items():
    prec, rec, _ = precision_recall_curve(test_reg.y, p)
    ax.plot(rec, prec, label=f"{name} (PR-AUC={average_precision_score(test_reg.y, p):.3f})")
ax.axhline(test_reg.y.mean(), linestyle="--", label="Prevalencia")
ax.set(xlabel="Recall", ylabel="Precisión", title="Curvas precisión–recall — test")
ax.legend(); plt.tight_layout(); plt.show()

## 9. Bootstrap por localidad del modelo con menor AIC de entrenamiento

La selección del modelo para bootstrap se basa en el AIC no ponderado calculado solo con entrenamiento, no en el test. En cada réplica se remuestrean localidades con reemplazo y se conservan sus registros de entrenamiento. Esto reconoce parcialmente el agrupamiento espacial, pero no resuelve la dependencia temporal dentro de cada localidad. La proporción de réplicas que mantienen un signo describe estabilidad empírica, no una probabilidad de que una hipótesis sea verdadera.

In [ ]:
aic_train = {name: fit_aic(train_reg, vars_).aic for name, vars_ in specs.items()}
best_name = min(aic_train, key=aic_train.get)
best_features = specs[best_name]
print("Modelo para bootstrap:", best_name, "| variables:", best_features)

N_BOOTSTRAP = 100  # Para el informe final, aumentar a 300+ si el tiempo lo permite.
rng = np.random.default_rng(SEED)
locs = train_reg["Location"].dropna().unique()
groups = {loc: train_reg[train_reg.Location == loc].copy() for loc in locs}
boot_rows = []
for b in range(N_BOOTSTRAP):
    sampled = rng.choice(locs, size=len(locs), replace=True)
    boot = pd.concat([groups[loc] for loc in sampled], ignore_index=True)
    try:
        fitted = fit_weighted(boot, best_features)
        for var, val in fitted.params.items():
            boot_rows.append({"replica": b, "variable": var, "coeficiente": float(val)})
    except Exception:
        continue

boot_df = pd.DataFrame(boot_rows)
if boot_df.empty:
    raise RuntimeError("No se completaron réplicas bootstrap.")
boot_summary = boot_df.groupby("variable").coeficiente.agg(
    n_replicas="count", media="mean", mediana="median",
    ic95_inf=lambda x: x.quantile(.025), ic95_sup=lambda x: x.quantile(.975),
    proporcion_positivo=lambda x: (x > 0).mean()
).reset_index()
boot_summary["proporcion_signo_mayoritario"] = np.maximum(
    boot_summary.proporcion_positivo, 1 - boot_summary.proporcion_positivo
)
display(boot_summary.round(4))

plot = boot_summary.sort_values("mediana").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9, max(4, .45*len(plot))))
ypos = np.arange(len(plot))
ax.errorbar(plot.mediana, ypos,
            xerr=[plot.mediana - plot.ic95_inf, plot.ic95_sup - plot.mediana],
            fmt="o", capsize=3)
ax.axvline(0, linestyle="--")
ax.set_yticks(ypos); ax.set_yticklabels(plot.variable)
ax.set_xlabel("Coeficiente (IC percentil 95 %)"); ax.set_title("Bootstrap por localidad")
plt.tight_layout(); plt.show()

## 10. Síntesis final (máximo una página)

La siguiente síntesis se genera con los resultados de la ejecución. Revísala antes de entregar: agrega interpretación sustantiva y no declares superior un modelo solo por accuracy.

In [ ]:
best_auc = metricas_test["roc_auc"].idxmax()
best_recall = metricas_test["recall_yes"].idxmax()
n_boot_ok = int(boot_df.replica.nunique())
sintesis = f'''
### Síntesis de resultados

Se ajustaron tres regresiones logísticas para `RainTomorrow`, utilizando una partición estratificada de 70 % entrenamiento y 30 % prueba. La clase positiva representa {df.y.mean():.2%} de los registros etiquetados, por lo que la exactitud debe complementarse con recall, precisión, F1, ROC-AUC y PR-AUC.

La comparación de faltantes se evaluó sobre una cohorte común de {len(test_complete):,} casos completos de prueba. La eliminación reduce los datos disponibles para entrenar; la imputación simple es transparente y conserva observaciones; la imputación por regresión estima `Sunshine` a partir de variables meteorológicas y localidad. Dado el alto porcentaje de faltantes de `Sunshine`, y que no se ha demostrado un mecanismo MCAR/MAR, las estimaciones imputadas deben interpretarse con cautela.

El mayor ROC-AUC en test correspondió a **{best_auc}** ({metricas_test.loc[best_auc, "roc_auc"]:.3f}); el mayor recall para `Yes` correspondió a **{best_recall}** ({metricas_test.loc[best_recall, "recall_yes"]:.3f}). La ponderación 2,2 prioriza la detección de lluvia, pero el umbral definitivo debe definirse con validación en entrenamiento y según el costo relativo de falsos positivos y falsos negativos.

El bootstrap por localidad completó {n_boot_ok} réplicas. La estabilidad se interpreta mediante los intervalos y la proporción de réplicas que mantienen el signo de cada coeficiente. Los resultados no demuestran causalidad y pueden verse afectados por dependencia temporal, diferencias entre localidades y exploración previa del conjunto completo.

Para S3 se recomienda validar el umbral sin usar test, probar una partición temporal (entrenar con años anteriores y evaluar años posteriores), documentar los costos de error y aumentar las réplicas bootstrap si el tiempo computacional lo permite.
'''
display(Markdown(sintesis))

## 11. Exportación de resultados
Se escriben salidas nuevas en `results/04_formativa2/`; no se modifican los archivos originales ni los resultados de S1/S2.

In [ ]:
out = Path("results/04_formativa2")
out.mkdir(parents=True, exist_ok=True)
comparacion_imputacion.to_csv(out / "comparacion_imputacion.csv")
vif.to_csv(out / "vif_predictores.csv", index=False)
aic_table.to_csv(out / "comparacion_modelos_aic.csv", index=False)
coeficientes.to_csv(out / "coeficientes_modelos.csv")
metricas_test.to_csv(out / "metricas_test.csv")
boot_summary.to_csv(out / "bootstrap_coeficientes.csv", index=False)
with open(out / "resumen_modelamiento.json", "w", encoding="utf-8") as f:
    json.dump({
        "semilla": SEED, "train": 0.70, "test": 0.30, "estratificada": True,
        "prevalencia_yes": float(df.y.mean()), "modelo_bootstrap": best_name,
        "variables_bootstrap": best_features, "bootstrap_solicitadas": N_BOOTSTRAP,
        "bootstrap_completadas": n_boot_ok
    }, f, ensure_ascii=False, indent=2)
print("Resultados guardados en:", out.resolve())
print(sorted(p.name for p in out.iterdir()))